# 3D Alignment Analysis Pipeline (Final Version V2)

## Overview
Complete pipeline for analyzing synthetic 3D data quality.

### Features:
1.  **Structured Excel Reporting:** 
    * File per Batch.
    * Sheet per Sensor (Multiple Positions stacked in one sheet).
    * Includes Transformation Matrices.
2.  **Smart Visualization:** 
    * Input via relative folder path (e.g. `"Test_1/cam_d435/setup_front"`).
    * `view_alignment_overlay`: Loads the calculated matrix from Excel.
    * `create_gif`: Generates a **2x2 Multi-View Animation** (Front, Right, Top, Iso).

---

In [20]:
# Install dependencies if missing
try:
    import openpyxl
except ImportError:
    !pip install openpyxl

try:
    import imageio
except ImportError:
    !pip install imageio

try:
    import cv2
except ImportError:
    !pip install opencv-python
    import cv2    


import open3d as o3d
import pandas as pd
import numpy as np
import os
import time
import math
import copy
import imageio
import ast
from IPython.display import display, Markdown

print(f"Open3D version: {o3d.__version__}")

Open3D version: 0.19.0


In [21]:
# --- CONFIGURATION ---
PROJECT_DIR = os.getcwd()
DATA_ROOT = os.path.join(PROJECT_DIR, "..", "Blender_Generated_Data\Cube")

# Settings
VOXEL_SIZE = 0.01  # Resolution for RANSAC downsampling

print(f"Data Root: {DATA_ROOT}")

Data Root: c:\Users\RobinSchool\Stichting Hogeschool Utrecht\MNLE Imagine Project - Documents\DataProgram\Program's\..\Blender_Generated_Data\Cube


## 1. Helper Functions

In [22]:
def load_csv_pcd(filepath):
    """Loads XYZ data from a CSV file into an Open3D PointCloud."""
    try:
        if not os.path.exists(filepath): return None
        df = pd.read_csv(filepath)
        if len(df) < 10: return None
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(df[['X', 'Y', 'Z']].values)
        return pcd
    except: return None

def parse_ground_truth_matrix(row):
    """Extracts 4x4 matrix from dataframe row."""
    try:
        cols = [f"m{r}{c}" for r in range(4) for c in range(4)]
        flat = row[cols].values.astype(float)
        return flat.reshape(4, 4)
    except:
        return np.eye(4)

def matrix_to_str(matrix):
    """Flattens a 4x4 matrix to a string for Excel storage."""
    return str(matrix.tolist())

def str_to_matrix(mat_str):
    """Parses a string back to a numpy matrix."""
    try:
        return np.array(ast.literal_eval(mat_str))
    except:
        return np.eye(4)

def get_angular_error(R_est, R_gt):
    R_diff = np.dot(R_est, R_gt.T)
    tr = np.clip(np.trace(R_diff), -1, 3)
    return math.degrees(math.acos((tr - 1) / 2))

def get_translation_error(t_est, t_gt):
    return np.linalg.norm(t_est - t_gt)

def preprocess_point_cloud(pcd, voxel_size):
    pcd_down = pcd.voxel_down_sample(voxel_size)
    if len(pcd_down.points) < 5: return None, None
    pcd_down.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 2, max_nn=30))
    try:
        pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
            pcd_down, o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100))
    except RuntimeError: return None, None
    return pcd_down, pcd_fpfh

## 2. Alignment Engine

In [23]:

def align_point_cloud(source_pcd, target_pcd, target_down, target_fpfh, voxel_size):
    """
    Voert een 3-traps alignment uit:
    1. RANSAC (Global): Op gedownsamplede data voor grove rotatie.
    2. ICP Grof (Local): Op gedownsamplede data voor snelle convergentie.
    3. ICP Fijn (Local): Op ORIGINELE data voor maximale precisie.
    """
    start_time = time.time()
    stats = {}
    
    # --- 1. PREPROCESS SOURCE (Low Res) ---
    source_down, source_fpfh = preprocess_point_cloud(source_pcd, voxel_size)
    if source_down is None: return None, None
    
    # --- 2. GLOBAL REGISTRATION (RANSAC) ---
    # Werkt op VOXEL data (snel & robuust tegen lokale minima)
    distance_threshold = voxel_size * 1.5
    
    ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh, True,
        distance_threshold,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False), 3, 
        [o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
         o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)],
        o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999)
    )
    time_ransac = time.time()
    
    # --- 3. ICP COARSE (Snel) ---
    # Werkt nog steeds op VOXEL data.
    # Dit trekt de wolken naar elkaar toe zonder zware berekeningen op miljoenen punten.
    icp_coarse_thresh = voxel_size * 0.4
    
    icp_coarse = o3d.pipelines.registration.registration_icp(
        source_down, target_down, icp_coarse_thresh, ransac.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane()
    )
    
    # --- 4. ICP FINE (Precies) ---
    # Werkt op de ORIGINELE (Full Res) data.
    # We gebruiken een heel kleine threshold omdat ze al bijna goed liggen.
    
    # We moeten eerst normals berekenen op de High-Res wolken (als ze die nog niet hebben)
    # Radius is kleiner omdat we fijnere details willen zien.
    radius_fine = voxel_size * 1.0 
    
    if not source_pcd.has_normals():
        source_pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=radius_fine, max_nn=30))
    if not target_pcd.has_normals():
        target_pcd.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=radius_fine, max_nn=30))
        
    icp_fine_thresh = voxel_size * 0.1 # Zeer strenge drempel (bijv. 5mm als voxel 5cm is)
    
    icp_fine = o3d.pipelines.registration.registration_icp(
        source_pcd, target_pcd, icp_fine_thresh, icp_coarse.transformation,
        o3d.pipelines.registration.TransformationEstimationPointToPlane(),
        o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=30)
    )
    
    end_time = time.time()
    
    # --- STATS ---
    stats['time_ransac'] = time_ransac - start_time
    stats['time_icp'] = end_time - time_ransac # Tijd van Coarse + Fine samen
    stats['time_total'] = end_time - start_time
    stats['fitness'] = icp_fine.fitness
    stats['rmse'] = icp_fine.inlier_rmse
    
    return icp_fine.transformation, stats

## 3. Analysis Pipeline (Excel Report Generation)

In [24]:
def save_batch_excel(batch_id, df_results):
    """Saves report. Structure: Overview | Sensor1 | Sensor2 ..."""
    filename = os.path.join(DATA_ROOT, f"Analysis_{batch_id}.xlsx")
    
    with pd.ExcelWriter(filename, engine='openpyxl') as writer:
        # 1. Overview Sheet
        overview = df_results.groupby(['Sensor', 'Position'])[['Fitness', 'Error_Rot_Deg', 'Error_Trans_M', 'Time_Total']].mean()
        overview.to_excel(writer, sheet_name='Overview')
        
        # 2. Hidden Raw Data (Useful for lookup)
        df_results.to_excel(writer, sheet_name='Raw_Data', index=False)
        
        # 3. Sheets per Sensor
        sensors = df_results['Sensor'].unique()
        for sensor in sensors:
            sheet_name = str(sensor)[:30]
            sensor_data = df_results[df_results['Sensor'] == sensor]
            positions = sensor_data['Position'].unique()
            
            row_cursor = 0
            for pos in positions:
                pos_data = sensor_data[sensor_data['Position'] == pos]
                
                # Header for Position Table
                pd.DataFrame([f"POSITION: {pos}"]).to_excel(writer, sheet_name=sheet_name, startrow=row_cursor, index=False, header=False)
                row_cursor += 1
                
                # Table
                cols = ['Sample_ID', 'Fitness', 'RMSE', 'Error_Rot_Deg', 'Error_Trans_M', 'Time_Total', 'Matrix_GT', 'Matrix_Est']
                pos_data[cols].to_excel(writer, sheet_name=sheet_name, startrow=row_cursor, index=False)
                
                row_cursor += len(pos_data) + 3 # Spacing
                
    print(f"   -> Report saved: {filename}")

def run_analysis_pipeline():
    print("Scanning...")
    if not os.path.exists(DATA_ROOT): return
    batches = [d for d in os.listdir(DATA_ROOT) if os.path.isdir(os.path.join(DATA_ROOT, d)) and d != "Output"]
    
    status_handle = display(Markdown("**Starting Analysis...**"), display_id=True)
    
    for batch_id in batches:
        batch_results = []
        batch_path = os.path.join(DATA_ROOT, batch_id)
        
        # Walk Sensors & Positions
        sensors = [d for d in os.listdir(batch_path) if os.path.isdir(os.path.join(batch_path, d))]
        for sens_id in sensors:
            s_path = os.path.join(batch_path, sens_id)
            positions = [d for d in os.listdir(s_path) if os.path.isdir(os.path.join(s_path, d))]
            
            for pos_id in positions:
                final_path = os.path.join(s_path, pos_id)
                gt_file = os.path.join(final_path, "ground_truth.csv")
                if not os.path.exists(gt_file): continue
                
                df_gt = pd.read_csv(gt_file, comment='#')
                if df_gt.empty: continue
                
                # Set Reference (Scan 0 of this folder)
                ref_file = df_gt.iloc[0]['filename']
                target_pcd = load_csv_pcd(os.path.join(final_path, ref_file))
                if target_pcd is None: continue
                target_down, target_fpfh = preprocess_point_cloud(target_pcd, VOXEL_SIZE)
                if target_down is None: continue

                status_handle.update(Markdown(f"**Processing:** {batch_id} | {sens_id} | {pos_id}"))
                
                for _, row in df_gt.iterrows():
                    if row['filename'] == ref_file: continue # Skip ref
                    
                    pcd = load_csv_pcd(os.path.join(final_path, row['filename']))
                    if pcd is None: continue
                    
                    est_matrix, stats = align_point_cloud(pcd, target_pcd, target_down, target_fpfh, VOXEL_SIZE)
                    if est_matrix is None: continue
                    
                    gt_matrix = parse_ground_truth_matrix(row)
                    try:
                        T_inv = np.linalg.inv(est_matrix)
                        rot_err = get_angular_error(T_inv[:3,:3], gt_matrix[:3,:3])
                        trans_err = get_translation_error(T_inv[:3,3], gt_matrix[:3,3])
                    except: rot_err, trans_err = 999, 999
                    
                    batch_results.append({
                        "Batch": batch_id, "Sensor": sens_id, "Position": pos_id,
                        "Sample_ID": row['sample_id'],
                        "Fitness": stats['fitness'], "RMSE": stats['rmse'],
                        "Error_Rot_Deg": rot_err, "Error_Trans_M": trans_err,
                        "Time_Total": stats['time_total'],
                        "Matrix_GT": matrix_to_str(gt_matrix),
                        "Matrix_Est": matrix_to_str(est_matrix)
                    })
        
        if batch_results:
            save_batch_excel(batch_id, pd.DataFrame(batch_results))
            
    status_handle.update(Markdown("### ✅ Analysis Complete!"))

In [25]:
# Run Pipeline
run_analysis_pipeline()

Scanning...


### ✅ Analysis Complete!

[Open3D WARNING] Too few correspondences (128) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (122) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (68) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (57) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (59) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (75) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (37) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (94) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (54) after mutual filter, fall back to original correspondences.
[Open3D WARNING] Too few correspondences (46

## 4. Advanced Visualization Functions

In [26]:
def resolve_scan_paths(scan_folder_rel, source_id, ref_id):
    """Finds paths and loads point clouds."""
    full_path = os.path.join(DATA_ROOT, scan_folder_rel)
    gt_path = os.path.join(full_path, "ground_truth.csv")
    
    if not os.path.exists(gt_path):
        print(f"Error: Path not found {gt_path}"); return None, None, None, None
        
    df_gt = pd.read_csv(gt_path, comment='#')
    
    # Find Files
    src_row = df_gt[df_gt['sample_id'] == source_id]
    ref_row = df_gt[df_gt['sample_id'] == ref_id]
    if src_row.empty or ref_row.empty: 
        print("Error: ID not found."); return None, None, None, None

    src_pcd = load_csv_pcd(os.path.join(full_path, src_row.iloc[0]['filename']))
    ref_pcd = load_csv_pcd(os.path.join(full_path, ref_row.iloc[0]['filename']))
    
    # Metadata for Excel lookup
    parts = scan_folder_rel.replace("\\", "/").split("/")
    batch = parts[0] if len(parts) > 0 else ""
    pos = parts[2] if len(parts) > 2 else ""
    
    return src_pcd, ref_pcd, batch, pos

def view_alignment_overlay(scan_folder, source_id, ref_id=0):
    """Shows static overlay using matrix from Excel."""
    src_pcd, ref_pcd, batch, pos = resolve_scan_paths(scan_folder, source_id, ref_id)
    if src_pcd is None: return

    # Look for Matrix in Excel
    excel_path = os.path.join(DATA_ROOT, f"Analysis_{batch}.xlsx")
    transform = np.eye(4)
    
    if os.path.exists(excel_path):
        try:
            df_raw = pd.read_excel(excel_path, sheet_name='Raw_Data')
            match = df_raw[(df_raw['Position'] == pos) & (df_raw['Sample_ID'] == source_id)]
            if not match.empty:
                transform = str_to_matrix(match.iloc[0]['Matrix_Est'])
                print("Loaded transformation from Excel.")
        except: pass

    # Viz
    ref_pcd.paint_uniform_color([0.6, 0.6, 0.6]) # Grey
    src_pcd.transform(transform)
    src_pcd.paint_uniform_color([1, 0, 0])       # Red
    
    o3d.visualization.draw_geometries([ref_pcd, src_pcd], window_name=f"Overlay ID {source_id}", width=1000, height=800)



In [27]:
view_alignment_overlay(r"Hard test\high_prec\setup_iso",2)
# C:\Users\RobinSchool\Stichting Hogeschool Utrecht\MNLE Imagine Project - Documents\DataProgram\Blender_Generated_Data\Cube\Hard test\high_prec\setup_iso

Loaded transformation from Excel.


In [28]:
def draw_text_overlay(image, text_lines, bg_color=(0, 0, 0), text_color=(255, 255, 255)):
    """
    Draws a black bar with multi-line white text at the top of the image.
    text_lines: Can be a single string or a list of strings.
    """
    if isinstance(text_lines, str):
        text_lines = [text_lines]
        
    h, w, _ = image.shape
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 1.0 # Slightly smaller to fit more info
    thickness = 2
    line_height = 40
    padding = 20
    
    # Calculate box height
    box_height = padding * 2 + len(text_lines) * line_height
    
    # Draw Background Bar
    cv2.rectangle(image, (0, 0), (w, box_height), bg_color, -1)
    
    # Draw Lines
    y = padding + 30
    for line in text_lines:
        (text_w, text_h), _ = cv2.getTextSize(line, font, font_scale, thickness)
        text_x = (w - text_w) // 2 # Center
        cv2.putText(image, line, (text_x, y), font, font_scale, text_color, thickness, cv2.LINE_AA)
        y += line_height
        
    return image



In [29]:
def create_presentation_gif(
    scan_folder, source_id, ref_id=0, 
    save_gif=True, save_mp4=True, fps=10, 
    pause_time=1.0 
):
    """
    Generates a storytelling animation.
    V5 Update: Saves as MP4 for WhatsApp compatibility.
    """
    # 1. Load Data
    src_pcd_high, ref_pcd_high, batch, pos = resolve_scan_paths(scan_folder, source_id, ref_id)
    if src_pcd_high is None: return

    # Load GT for final stats
    full_path = os.path.join(DATA_ROOT, scan_folder)
    gt_path = os.path.join(full_path, "ground_truth.csv")
    df_gt = pd.read_csv(gt_path, comment='#')
    gt_row = df_gt[df_gt['sample_id'] == source_id].iloc[0]
    gt_matrix = parse_ground_truth_matrix(gt_row)

    print(f"Generating Animation for {scan_folder} (ID {source_id})...")
    
    # Colors
    ref_color = [0.6, 0.6, 0.6] # Grey
    src_color = [1.0, 0.0, 0.0] # Red
    
    ref_pcd_high.paint_uniform_color(ref_color)
    src_pcd_high.paint_uniform_color(src_color)
    
    # 2. Pre-calculations
    # Create Low-Res versions
    src_down, s_fpfh = preprocess_point_cloud(src_pcd_high, VOXEL_SIZE)
    ref_down, t_fpfh = preprocess_point_cloud(ref_pcd_high, VOXEL_SIZE)
    
    src_down.paint_uniform_color(src_color)
    ref_down.paint_uniform_color(ref_color)
    
    # Calculate RANSAC Target
    ransac = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        src_down, ref_down, s_fpfh, t_fpfh, True, VOXEL_SIZE * 1.5,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False), 3, 
        [o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.9),
         o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(VOXEL_SIZE * 1.5)],
        o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999)
    )
    ransac_final_T = ransac.transformation
    
    # 3. Visualization Setup
    # Note: 1000x1000 is good, usually safe for MP4 macroblocks (divisible by 2)
    vis = o3d.visualization.Visualizer()
    vis.create_window(width=1000, height=1000, visible=True)
    
    vis.add_geometry(ref_pcd_high)
    vis.add_geometry(src_pcd_high)
    ctr = vis.get_view_control()
    
    # --- Capture Helper ---
    def capture_frame(text_lines):
        views = [
            {'lookat': [0,0,0], 'front': [0, -1, 0], 'up': [0, 0, 1]}, 
            {'lookat': [0,0,0], 'front': [1, 0, 0],  'up': [0, 0, 1]}, 
            {'lookat': [0,0,0], 'front': [0, 0, 1],  'up': [0, 1, 0]}, 
            {'lookat': [0,0,0], 'front': [1, -1, 1], 'up': [0, 0, 1]} 
        ]
        imgs = []
        for v in views:
            ctr.set_lookat(v['lookat'])
            ctr.set_front(v['front'])
            ctr.set_up(v['up'])
            ctr.set_zoom(0.85)  # Wide Zoom
            vis.poll_events()
            vis.update_renderer()
            buf = vis.capture_screen_float_buffer(False)
            imgs.append((255 * np.asarray(buf)).astype(np.uint8))
            
        top = np.hstack((imgs[0], imgs[1]))
        bot = np.hstack((imgs[2], imgs[3]))
        full_img = np.vstack((top, bot))
        
        # Add Overlay
        full_img = cv2.cvtColor(full_img, cv2.COLOR_RGB2BGR)
        full_img = draw_text_overlay(full_img, text_lines)
        full_img = cv2.cvtColor(full_img, cv2.COLOR_BGR2RGB)
        return full_img

    frames = []
    
    def add_pause(frames, last_frame, seconds):
        for _ in range(int(seconds * fps)):
            frames.append(last_frame)

    # ==========================
    # PHASE 1: DOWNSAMPLING
    # ==========================
    frame = capture_frame("1. Input Data (High Res)")
    frames.append(frame)
    add_pause(frames, frame, pause_time)
    
    # Switch to Low Res
    params = ctr.convert_to_pinhole_camera_parameters()
    vis.remove_geometry(src_pcd_high, reset_bounding_box=False)
    vis.remove_geometry(ref_pcd_high, reset_bounding_box=False)
    vis.add_geometry(src_down, reset_bounding_box=False)
    vis.add_geometry(ref_down, reset_bounding_box=False)
    ctr.convert_from_pinhole_camera_parameters(params)
    
    frame = capture_frame(f"1. Downsampling (Voxel: {VOXEL_SIZE}m)")
    frames.append(frame)
    add_pause(frames, frame, pause_time)
    
    # ==========================
    # PHASE 2: RANSAC
    # ==========================
    current_T = np.eye(4) 
    
    attempts = []
    for i in range(9):
        random_T = np.eye(4)
        random_T[:3, :3] = src_pcd_high.get_rotation_matrix_from_xyz(np.random.uniform(-1, 1, 3))
        random_T[:3, 3] = np.random.uniform(-0.5, 0.5, 3)
        attempts.append((random_T, f"2. RANSAC Search (Attempt {i+1}/10)"))

    attempts.append((ransac_final_T, "2. RANSAC Found (Attempt 10: Best Match)"))

    for target_T, label in attempts:
        delta_T = np.dot(target_T, np.linalg.inv(current_T))
        src_down.transform(delta_T)
        vis.update_geometry(src_down)
        current_T = target_T 
        
        frame = capture_frame([label, "Typical: 100k+ Iterations"])
        frames.append(frame)
        add_pause(frames, frame, 0.5)

    add_pause(frames, frames[-1], 1.0)
    
    # ==========================
    # PHASE 3a: COARSE ICP
    # ==========================
    src_down.estimate_normals()
    ref_down.estimate_normals()
    
    for i in range(15):
        reg = o3d.pipelines.registration.registration_icp(
            src_down, ref_down, VOXEL_SIZE * 0.4, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPlane(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1)
        )
        src_down.transform(reg.transformation)
        vis.update_geometry(src_down)
        current_T = np.dot(reg.transformation, current_T)
        
        frame = capture_frame(f"3a. Coarse ICP (Low Res) - Iter {i+1}")
        frames.append(frame)
        
    add_pause(frames, frames[-1], 1.0)
    
    # ==========================
    # PHASE 3b: FINE ICP
    # ==========================
    src_pcd_high.transform(current_T)
    
    params = ctr.convert_to_pinhole_camera_parameters()
    vis.remove_geometry(src_down, reset_bounding_box=False)
    vis.remove_geometry(ref_down, reset_bounding_box=False)
    vis.add_geometry(src_pcd_high, reset_bounding_box=False)
    vis.add_geometry(ref_pcd_high, reset_bounding_box=False)
    ctr.convert_from_pinhole_camera_parameters(params)
    
    frame = capture_frame("3b. Switch to High Res for Fine Tuning")
    frames.append(frame)
    add_pause(frames, frame, pause_time)
    
    radius_fine = VOXEL_SIZE * 2.0 
    src_pcd_high.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=radius_fine, max_nn=30))
    ref_pcd_high.estimate_normals(o3d.geometry.KDTreeSearchParamHybrid(radius=radius_fine, max_nn=30))
    
    threshold_fine = VOXEL_SIZE * 0.4
    
    for i in range(15):
        reg = o3d.pipelines.registration.registration_icp(
            src_pcd_high, ref_pcd_high, threshold_fine, np.eye(4),
            o3d.pipelines.registration.TransformationEstimationPointToPlane(),
            o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=1)
        )
        src_pcd_high.transform(reg.transformation)
        vis.update_geometry(src_pcd_high)
        current_T = np.dot(reg.transformation, current_T)
        
        frame = capture_frame(f"3b. Fine ICP (High Res) - Iter {i+1}")
        frames.append(frame)
        
    # ==========================
    # PHASE 4: FINAL STATISTICS
    # ==========================
    try:
        T_inv = np.linalg.inv(current_T)
        rot_err = get_angular_error(T_inv[:3,:3], gt_matrix[:3,:3])
        trans_err = get_translation_error(T_inv[:3,3], gt_matrix[:3,3])
    except:
        rot_err, trans_err = 999.0, 999.0
        
    mat_str = np.array2string(current_T, precision=3, suppress_small=True, separator=', ')
    stats_text = [
        "ALIGNMENT COMPLETE",
        f"Rotation Error: {rot_err:.4f} deg",
        f"Translation Error: {trans_err:.6f} m",
        "Final Matrix:",
        mat_str.split('\n')[0], mat_str.split('\n')[1], mat_str.split('\n')[2], mat_str.split('\n')[3]
    ]
    final_frame = capture_frame(stats_text)
    
    for _ in range(int(10.0 * fps)):
        frames.append(final_frame)

    vis.destroy_window()
    
    # --- SAVE OPTIONS ---
    if frames:
        # Option 1: Save as GIF (Looping)
        if save_gif:
            gif_path = os.path.join(DATA_ROOT, scan_folder, f"presentation_final_{source_id}.gif")
            imageio.mimsave(gif_path, frames, fps=fps, loop=0) # loop=0 is vital for WhatsApp
            print(f"GIF Saved: {gif_path}")
            
        # Option 2: Save as MP4 (Best for WhatsApp)
        if save_mp4:
            mp4_path = os.path.join(DATA_ROOT, scan_folder, f"presentation_final_{source_id}.mp4")
            # Quality=8 is high quality, macro_block_size=None helps with sizing issues
            imageio.mimsave(mp4_path, frames, fps=fps, quality=8, macro_block_size=None)
            print(f"VIDEO Saved (Use for WhatsApp): {mp4_path}")
        
        # Display (GIF is easier to display in Notebook)
        from IPython.display import Image
        if save_gif:
            display(Image(filename=gif_path))

In [30]:
create_presentation_gif("Test_1/high_prec/setup_iso", source_id=3)

Error: Path not found c:\Users\RobinSchool\Stichting Hogeschool Utrecht\MNLE Imagine Project - Documents\DataProgram\Program's\..\Blender_Generated_Data\Cube\Test_1/high_prec/setup_iso\ground_truth.csv
